In [14]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import plotly.graph_objects as go

from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.arima.model import ARIMA

from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error

In [15]:
data = pd.read_csv(
    "market_data_eda_features.csv",
    parse_dates=True,
    index_col="Date"
)

data.head()

,Close_AAPL,Close_MSFT,Close_SPY,High_AAPL,High_MSFT,High_SPY,Low_AAPL,Low_MSFT,Low_SPY,Open_AAPL,...,SMA_200_AAPL,Rolling_STD_AAPL,BB_Upper_AAPL,BB_Lower_AAPL,Daily_Return_AAPL,Month,Day_Name,Daily_Return_MSFT,Daily_Return_SPY,Rolling_Volatility_AAPL
Date,,,,,,,,,,,,,,,,,,,,,
2021-05-20,124.095665,237.058350,387.864471,124.495317,238.472160,389.125354,121.941464,234.538503,384.492806,122.068187,...,NaN,NaN,NaN,NaN,0.021012,5,Thursday,0.013820,0.010758,NaN
2021-05-21,122.263161,235.798447,387.546844,124.768273,238.837660,390.591638,122.048714,235.384890,387.089202,124.592818,...,NaN,NaN,NaN,NaN,-0.014767,5,Friday,-0.005315,-0.000819,NaN
2021-05-24,123.890968,241.194031,391.497620,124.709764,241.559510,392.571694,122.760260,238.049021,389.545570,122.828492,...,NaN,NaN,NaN,NaN,0.013314,5,Monday,0.022882,0.010194,NaN
2021-05-25,123.696022,242.098083,390.629089,125.080175,243.088711,392.936029,123.130664,241.232492,390.050024,124.592792,...,NaN,NaN,NaN,NaN,-0.001574,5,Tuesday,0.003748,-0.002218,NaN
2021-05-26,123.647278,241.876892,391.404297,124.173645,243.271463,391.908627,123.228134,241.165173,390.180781,123.754501,...,NaN,NaN,NaN,NaN,-0.000394,5,Wednesday,-0.000914,0.001985,NaN


In [16]:
def adf_test(series, title=""):
    result = adfuller(series.dropna())

    print(f"ADF Test: {title}")
    print("-" * 40)
    print(f"ADF Statistic: {result[0]:.4f}")
    print(f"p-value: {result[1]:.4f}")
    print("Critical Values:")

    for key, value in result[4].items():
        print(f"   {key}: {value:.4f}")

    if result[1] <= 0.05:
        print("Result: Stationary")
    else:
        print("Result: Non-stationary")

In [17]:
adf_test(data["Close_AAPL"], "Apple Closing Price")

ADF Test: Apple Closing Price
----------------------------------------
ADF Statistic: -0.6440
p-value: 0.8607
Critical Values:
   1%: -3.4356
   5%: -2.8639
   10%: -2.5680
Result: Non-stationary


In [18]:
adf_test(data["Daily_Return_AAPL"], "Apple Daily Returns")

ADF Test: Apple Daily Returns
----------------------------------------
ADF Statistic: -34.7603
p-value: 0.0000
Critical Values:
   1%: -3.4356
   5%: -2.8639
   10%: -2.5680
Result: Stationary


In [19]:
target = data["Close_AAPL"].dropna()

In [20]:
split_index = int(len(target) * 0.8)

train = target.iloc[:split_index]
test = target.iloc[split_index:]

In [21]:
print("Train size:", len(train))
print("Test size:", len(test))

print("Train period:", train.index.min(), "to", train.index.max())
print("Test period:", test.index.min(), "to", test.index.max())

Train size: 1000
Test size: 251
Train period: 2021-05-20 00:00:00 to 2025-05-13 00:00:00
Test period: 2025-05-14 00:00:00 to 2026-05-13 00:00:00


In [22]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=train.index,
        y=train,
        mode="lines",
        name="Training Data"
    )
)

fig.add_trace(
    go.Scatter(
        x=test.index,
        y=test,
        mode="lines",
        name="Testing Data"
    )
)

fig.update_layout(
    title="Apple Closing Price Train/Test Split",
    xaxis_title="Date",
    yaxis_title="Price (USD)",
    template="plotly_dark"
)

fig.show()

In [23]:
arima_model = ARIMA(train.values, order=(5, 1, 0))
arima_result = arima_model.fit()

print(arima_result.summary())

                               SARIMAX Results                                
Dep. Variable:                      y   No. Observations:                 1000
Model:                 ARIMA(5, 1, 0)   Log Likelihood               -2554.872
Date:                Sat, 16 May 2026   AIC                           5121.744
Time:                        17:04:15   BIC                           5151.185
Sample:                             0   HQIC                          5132.934
                               - 1000                                         
Covariance Type:                  opg                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
ar.L1          0.0270      0.019      1.450      0.147      -0.010       0.064
ar.L2          0.0212      0.026      0.816      0.415      -0.030       0.072
ar.L3         -0.0518      0.024     -2.119      0.0

In [24]:
arima_forecast_values = arima_result.forecast(steps=len(test))

arima_forecast = pd.Series(
    arima_forecast_values,
    index=test.index
)

Traditional ARIMA models capture short-term structure but struggle with highly volatile equity price movements.

In [25]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=train.index,
        y=train,
        mode="lines",
        name="Training Data"
    )
)

fig.add_trace(
    go.Scatter(
        x=test.index,
        y=test,
        mode="lines",
        name="Actual Test Data"
    )
)

fig.add_trace(
    go.Scatter(
        x=test.index,
        y=arima_forecast,
        mode="lines",
        name="ARIMA Forecast"
    )
)

fig.update_layout(
    title="ARIMA Forecast vs Actual Apple Closing Price",
    xaxis_title="Date",
    yaxis_title="Price (USD)",
    template="plotly_dark"
)

fig.show()

The ARIMA model produced a relatively flat forecast compared to the actual test-period movement. This suggests that while ARIMA can capture short-term historical dependence, it struggles to model sudden trend shifts and strong upward momentum in equity prices. The result highlights the limitation of purely statistical models for financial forecasting, especially when external market drivers are not included.

In [26]:
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error
import numpy as np

rmse = np.sqrt(mean_squared_error(test, arima_forecast))
mape = mean_absolute_percentage_error(test, arima_forecast) * 100

print("ARIMA Model Evaluation")
print("----------------------")
print(f"RMSE: {rmse:.2f}")
print(f"MAPE: {mape:.2f}%")

ARIMA Model Evaluation
----------------------
RMSE: 45.13
MAPE: 14.94%


The ARIMA(5,1,0) model produced a test-set RMSE of 45.13 and MAPE of 14.94%, indicating moderate forecasting error. The model captured the general price level but failed to follow the strong upward movement in the test period, showing the limitation of univariate statistical models for volatile stock price forecasting.

In [27]:
!pip install prophet -q

In [28]:
from prophet import Prophet

In [29]:
prophet_data = data[["Close_AAPL"]].reset_index()

prophet_data = prophet_data.rename(
    columns={
        "Date": "ds",
        "Close_AAPL": "y"
    }
)

prophet_data.head()

,ds,y
0,2021-05-20,124.095665
1,2021-05-21,122.263161
2,2021-05-24,123.890968
3,2021-05-25,123.696022
4,2021-05-26,123.647278


In [30]:
split_index = int(len(prophet_data) * 0.8)

prophet_train = prophet_data.iloc[:split_index]
prophet_test = prophet_data.iloc[split_index:]

print("Training data shape:", prophet_train.shape)
print("Testing data shape:", prophet_test.shape)

Training data shape: (1000, 2)
Testing data shape: (251, 2)


In [31]:
prophet_model = Prophet(
    daily_seasonality=False,
    weekly_seasonality=True,
    yearly_seasonality=True
)

prophet_model.fit(prophet_train)

In [32]:
future_test = prophet_test[["ds"]]

prophet_forecast = prophet_model.predict(future_test)

prophet_forecast[["ds", "yhat", "yhat_lower", "yhat_upper"]].head()

,ds,yhat,yhat_lower,yhat_upper
0,2025-05-14,209.041642,199.750889,217.930163
1,2025-05-15,208.774514,198.924841,217.537619
2,2025-05-16,208.861524,200.080851,217.076497
3,2025-05-19,208.807664,200.276683,217.630642
4,2025-05-20,208.624609,199.643976,217.912494


In [33]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=prophet_train["ds"],
        y=prophet_train["y"],
        mode="lines",
        name="Training Data"
    )
)

fig.add_trace(
    go.Scatter(
        x=prophet_test["ds"],
        y=prophet_test["y"],
        mode="lines",
        name="Actual Test Data"
    )
)

fig.add_trace(
    go.Scatter(
        x=prophet_forecast["ds"],
        y=prophet_forecast["yhat"],
        mode="lines",
        name="Prophet Forecast"
    )
)

fig.add_trace(
    go.Scatter(
        x=prophet_forecast["ds"],
        y=prophet_forecast["yhat_upper"],
        mode="lines",
        name="Upper Confidence Interval",
        line=dict(width=0),
        showlegend=False
    )
)

fig.add_trace(
    go.Scatter(
        x=prophet_forecast["ds"],
        y=prophet_forecast["yhat_lower"],
        mode="lines",
        name="Lower Confidence Interval",
        fill="tonexty",
        line=dict(width=0),
        showlegend=True
    )
)

fig.update_layout(
    title="Prophet Forecast vs Actual Apple Closing Price",
    xaxis_title="Date",
    yaxis_title="Price (USD)",
    template="plotly_dark"
)

fig.show()

In [34]:
prophet_rmse = np.sqrt(
    mean_squared_error(prophet_test["y"], prophet_forecast["yhat"])
)

prophet_mape = mean_absolute_percentage_error(
    prophet_test["y"], prophet_forecast["yhat"]
) * 100

print("Prophet Model Evaluation")
print("------------------------")
print(f"RMSE: {prophet_rmse:.2f}")
print(f"MAPE: {prophet_mape:.2f}%")

Prophet Model Evaluation
------------------------
RMSE: 33.09
MAPE: 10.91%


### Prophet Model Interpretation

The Prophet model achieved an RMSE of 33.09 and MAPE of 10.91%, outperforming the ARIMA(5,1,0) model, which recorded an RMSE of 45.13 and MAPE of 14.94%.

Compared to ARIMA, Prophet captured the upward trend in the test period more effectively. This is because Prophet is designed to model trend changes, seasonality, and changepoints, while ARIMA relies mainly on past values and differencing.

However, the forecast still shows uncertainty, especially toward the later part of the test period. This indicates that stock prices remain difficult to predict using historical price data alone. External factors such as earnings announcements, interest rates, market sentiment, and macroeconomic events may be needed to improve forecast reliability.

# Tuned Prophet Model

In [35]:
param_grid = {
    "changepoint_prior_scale": [0.01, 0.05, 0.1, 0.5],
    "seasonality_prior_scale": [1, 5, 10],
    "seasonality_mode": ["additive", "multiplicative"]
}

In [36]:
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error
import numpy as np
import pandas as pd

results = []

for cps in param_grid["changepoint_prior_scale"]:
    for sps in param_grid["seasonality_prior_scale"]:
        for mode in param_grid["seasonality_mode"]:

            model = Prophet(
                daily_seasonality=False,
                weekly_seasonality=True,
                yearly_seasonality=True,
                changepoint_prior_scale=cps,
                seasonality_prior_scale=sps,
                seasonality_mode=mode
            )

            model.fit(prophet_train)

            forecast = model.predict(prophet_test[["ds"]])

            rmse_score = np.sqrt(
                mean_squared_error(prophet_test["y"], forecast["yhat"])
            )

            mape_score = mean_absolute_percentage_error(
                prophet_test["y"], forecast["yhat"]
            ) * 100

            results.append({
                "changepoint_prior_scale": cps,
                "seasonality_prior_scale": sps,
                "seasonality_mode": mode,
                "RMSE": rmse_score,
                "MAPE (%)": mape_score
            })

tuning_results = pd.DataFrame(results)

tuning_results.sort_values("MAPE (%)").head(10)

,changepoint_prior_scale,seasonality_prior_scale,seasonality_mode,RMSE,MAPE (%)
19,0.50,1,multiplicative,23.359400,7.278666
4,0.01,10,additive,21.280289,7.431190
2,0.01,5,additive,21.263226,7.432080
0,0.01,1,additive,21.331306,7.440020
5,0.01,10,multiplicative,23.606603,7.922341
21,0.50,5,multiplicative,25.181307,8.215372
17,0.10,10,multiplicative,27.772774,8.635619
15,0.10,5,multiplicative,28.003363,8.786639
1,0.01,1,multiplicative,27.235620,9.149523
3,0.01,5,multiplicative,27.753639,9.367483


### Tuned Prophet Model Interpretation

Hyperparameter tuning significantly improved the Prophet model's forecasting performance. The best configuration used a changepoint prior scale of 0.50, seasonality prior scale of 1, and multiplicative seasonality.

The tuned Prophet model achieved an RMSE of 23.36 and MAPE of 7.28%, outperforming both the baseline ARIMA and baseline Prophet models. This indicates that allowing the model greater flexibility to detect trend changes helped it better capture Apple’s upward price movement during the test period.

The improvement also suggests that model tuning is important in financial forecasting, as default parameters may not adequately capture rapid market trend shifts.

In [37]:
best_prophet_model = Prophet(
    daily_seasonality=False,
    weekly_seasonality=True,
    yearly_seasonality=True,
    changepoint_prior_scale=0.50,
    seasonality_prior_scale=1,
    seasonality_mode="multiplicative"
)

best_prophet_model.fit(prophet_train)

best_forecast = best_prophet_model.predict(prophet_test[["ds"]])

best_prophet_rmse = np.sqrt(
    mean_squared_error(prophet_test["y"], best_forecast["yhat"])
)

best_prophet_mape = mean_absolute_percentage_error(
    prophet_test["y"], best_forecast["yhat"]
) * 100

print("Tuned Prophet Model Evaluation")
print("------------------------------")
print(f"RMSE: {best_prophet_rmse:.2f}")
print(f"MAPE: {best_prophet_mape:.2f}%")

Tuned Prophet Model Evaluation
------------------------------
RMSE: 23.36
MAPE: 7.28%


In [38]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=prophet_train["ds"],
        y=prophet_train["y"],
        mode="lines",
        name="Training Data"
    )
)

fig.add_trace(
    go.Scatter(
        x=prophet_test["ds"],
        y=prophet_test["y"],
        mode="lines",
        name="Actual Test Data"
    )
)

fig.add_trace(
    go.Scatter(
        x=best_forecast["ds"],
        y=best_forecast["yhat"],
        mode="lines",
        name="Tuned Prophet Forecast"
    )
)

fig.update_layout(
    title="Tuned Prophet Forecast vs Actual Apple Closing Price",
    xaxis_title="Date",
    yaxis_title="Price (USD)",
    template="plotly_dark"
)

fig.show()

In [39]:
model_comparison = pd.DataFrame({
    "Model": ["ARIMA(5,1,0)", "Baseline Prophet", "Tuned Prophet"],
    "RMSE": [rmse, prophet_rmse, best_prophet_rmse],
    "MAPE (%)": [mape, prophet_mape, best_prophet_mape]
})

model_comparison

,Model,RMSE,MAPE (%)
0,"ARIMA(5,1,0)",45.131237,14.941747
1,Baseline Prophet,33.086158,10.910322
2,Tuned Prophet,23.359400,7.278666


Although the lowest MAPE was achieved by a highly flexible multiplicative Prophet model, its confidence interval became excessively wide, indicating unstable uncertainty estimates. Therefore, a more conservative tuned Prophet configuration was selected for final reporting because it provided a better balance between accuracy, interpretability, and forecast stability.

In [40]:
stable_prophet_model = Prophet(
    daily_seasonality=False,
    weekly_seasonality=True,
    yearly_seasonality=True,
    changepoint_prior_scale=0.01,
    seasonality_prior_scale=10,
    seasonality_mode="additive"
)

stable_prophet_model.fit(prophet_train)

stable_forecast = stable_prophet_model.predict(prophet_test[["ds"]])

stable_prophet_rmse = np.sqrt(
    mean_squared_error(prophet_test["y"], stable_forecast["yhat"])
)

stable_prophet_mape = mean_absolute_percentage_error(
    prophet_test["y"], stable_forecast["yhat"]
) * 100

print("Stable Tuned Prophet Model Evaluation")
print("-------------------------------------")
print(f"RMSE: {stable_prophet_rmse:.2f}")
print(f"MAPE: {stable_prophet_mape:.2f}%")

Stable Tuned Prophet Model Evaluation
-------------------------------------
RMSE: 21.28
MAPE: 7.43%


In [41]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=prophet_train["ds"],
        y=prophet_train["y"],
        mode="lines",
        name="Training Data"
    )
)

fig.add_trace(
    go.Scatter(
        x=prophet_test["ds"],
        y=prophet_test["y"],
        mode="lines",
        name="Actual Test Data"
    )
)

fig.add_trace(
    go.Scatter(
        x=stable_forecast["ds"],
        y=stable_forecast["yhat"],
        mode="lines",
        name="Stable Tuned Prophet Forecast"
    )
)

fig.add_trace(
    go.Scatter(
        x=stable_forecast["ds"],
        y=stable_forecast["yhat_upper"],
        mode="lines",
        line=dict(width=0),
        showlegend=False
    )
)

fig.add_trace(
    go.Scatter(
        x=stable_forecast["ds"],
        y=stable_forecast["yhat_lower"],
        mode="lines",
        fill="tonexty",
        line=dict(width=0),
        name="Confidence Interval"
    )
)

fig.update_layout(
    title="Stable Tuned Prophet Forecast vs Actual Apple Closing Price",
    xaxis_title="Date",
    yaxis_title="Price (USD)",
    template="plotly_dark"
)

fig.show()

### Stable Tuned Prophet Model Interpretation

The stable tuned Prophet model provides a more reliable forecast compared to the highly flexible tuned model. While the earlier tuned model achieved slightly lower MAPE, its confidence interval became excessively wide, making the forecast less interpretable and less suitable for practical reporting.

This stable configuration uses a conservative changepoint setting and additive seasonality, allowing the model to capture the broader upward trend without overreacting to short-term fluctuations.

The forecast follows the general direction of the actual Apple closing price during the test period, although it still smooths out sudden market jumps. This shows that Prophet is useful for capturing medium-term trend behavior, but it may not fully capture sharp stock price movements caused by market news, earnings, or investor sentiment.

In [42]:
model_comparison = pd.DataFrame({
    "Model": [
        "ARIMA(5,1,0)",
        "Baseline Prophet",
        "Flexible Tuned Prophet",
        "Stable Tuned Prophet"
    ],
    "RMSE": [
        rmse,
        prophet_rmse,
        best_prophet_rmse,
        stable_prophet_rmse
    ],
    "MAPE (%)": [
        mape,
        prophet_mape,
        best_prophet_mape,
        stable_prophet_mape
    ]
})

model_comparison = model_comparison.round(2)
model_comparison

,Model,RMSE,MAPE (%)
0,"ARIMA(5,1,0)",45.13,14.94
1,Baseline Prophet,33.09,10.91
2,Flexible Tuned Prophet,23.36,7.28
3,Stable Tuned Prophet,21.28,7.43


In [43]:
model_comparison.to_csv("model_comparison.csv", index=False)

In [45]:
from google.colab import files

files.download("model_comparison.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Final Model Selection

The Stable Tuned Prophet model was selected as the final forecasting model. Although the Flexible Tuned Prophet model achieved the lowest MAPE, its confidence interval became excessively wide, making the forecast less stable and harder to interpret.

The Stable Tuned Prophet model achieved the lowest RMSE of 21.28 and a competitive MAPE of 7.43%. This indicates that it produced the smallest average error in price terms while maintaining better forecast stability and interpretability.

Therefore, the Stable Tuned Prophet model provides the best overall balance between accuracy, reliability, and practical usability for this project.

In [53]:
forecast_actual_vs_predicted = pd.DataFrame({
    "Date": prophet_test["ds"].values,
    "Actual Price": prophet_test["y"].values,
    "Predicted Price": stable_forecast["yhat"].values
})

forecast_actual_vs_predicted = forecast_actual_vs_predicted.round(2)

forecast_actual_vs_predicted.head()

,Date,Actual Price,Predicted Price
0,2025-05-14,211.49,226.79
1,2025-05-15,210.61,226.80
2,2025-05-16,210.43,227.22
3,2025-05-19,207.96,228.30
4,2025-05-20,206.04,228.07


In [54]:
forecast_actual_vs_predicted.to_csv(
    "forecast_actual_vs_predicted.csv",
    index=False
)

from google.colab import files

files.download("forecast_actual_vs_predicted.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# 90-Day Future Forecast

In [46]:
final_prophet_data = data[["Close_AAPL"]].reset_index()

final_prophet_data = final_prophet_data.rename(
    columns={
        "Date": "ds",
        "Close_AAPL": "y"
    }
)

final_prophet_data.head()

,ds,y
0,2021-05-20,124.095665
1,2021-05-21,122.263161
2,2021-05-24,123.890968
3,2021-05-25,123.696022
4,2021-05-26,123.647278


In [47]:
final_model = Prophet(
    daily_seasonality=False,
    weekly_seasonality=True,
    yearly_seasonality=True,
    changepoint_prior_scale=0.01,
    seasonality_prior_scale=10,
    seasonality_mode="additive"
)

final_model.fit(final_prophet_data)

In [48]:
# Create future dates for the next 90 business/trading days
future_90 = final_model.make_future_dataframe(
    periods=90,
    freq="B"
)

# Generate forecast
future_forecast_90 = final_model.predict(future_90)

# Keep only the future 90 days
future_only_90 = future_forecast_90.tail(90)

future_only_90[["ds", "yhat", "yhat_lower", "yhat_upper"]].head()

,ds,yhat,yhat_lower,yhat_upper
1251,2026-05-14,262.822057,244.481600,280.597348
1252,2026-05-15,262.777049,244.133856,281.160944
1253,2026-05-18,262.042628,243.225786,280.760487
1254,2026-05-19,261.354317,244.376606,278.976938
1255,2026-05-20,261.032958,243.479726,279.393425


In [49]:
# Use last 252 trading days as recent history
recent_history = final_prophet_data.tail(252)

fig = go.Figure()

# Recent historical price
fig.add_trace(
    go.Scatter(
        x=recent_history["ds"],
        y=recent_history["y"],
        mode="lines",
        name="Recent Historical Price"
    )
)

# 90-day future forecast
fig.add_trace(
    go.Scatter(
        x=future_only_90["ds"],
        y=future_only_90["yhat"],
        mode="lines",
        name="90-Day Future Forecast"
    )
)

# Upper confidence interval
fig.add_trace(
    go.Scatter(
        x=future_only_90["ds"],
        y=future_only_90["yhat_upper"],
        mode="lines",
        line=dict(width=0),
        showlegend=False
    )
)

# Lower confidence interval with shaded area
fig.add_trace(
    go.Scatter(
        x=future_only_90["ds"],
        y=future_only_90["yhat_lower"],
        mode="lines",
        fill="tonexty",
        line=dict(width=0),
        name="Forecast Confidence Interval"
    )
)

fig.update_layout(
    title="90-Day Future Apple Stock Price Forecast for 2026",
    xaxis_title="Date",
    yaxis_title="Price (USD)",
    template="plotly_dark"
)

fig.show()

### 90-Day Future Forecast Interpretation

This chart shows the final Stable Tuned Prophet model forecasting Apple stock prices for the next 90 business days beyond the available historical dataset.

The recent historical price trend is shown in blue, while the forecasted price path is shown in red. The purple shaded region represents the forecast confidence interval, which captures the expected uncertainty around the prediction.

The model projects a generally stable-to-upward short-term trend for Apple stock during the upcoming 2026 forecast period. However, the confidence interval widens as the forecast moves further into the future, showing that uncertainty increases over longer forecasting horizons.

This forecast should be interpreted as a data-driven estimate rather than a guaranteed market prediction. Stock prices can change due to earnings announcements, macroeconomic conditions, interest rates, investor sentiment, and unexpected market events.

In [50]:
future_forecast_summary = future_only_90[["ds", "yhat", "yhat_lower", "yhat_upper"]].copy()

future_forecast_summary.columns = [
    "Date",
    "Predicted Price",
    "Lower Confidence Bound",
    "Upper Confidence Bound"
]

future_forecast_summary = future_forecast_summary.round(2)

future_forecast_summary.head(10)

,Date,Predicted Price,Lower Confidence Bound,Upper Confidence Bound
1251,2026-05-14,262.82,244.48,280.60
1252,2026-05-15,262.78,244.13,281.16
1253,2026-05-18,262.04,243.23,280.76
1254,2026-05-19,261.35,244.38,278.98
1255,2026-05-20,261.03,243.48,279.39
1256,2026-05-21,260.33,242.96,277.86
1257,2026-05-22,260.35,242.26,278.25
1258,2026-05-25,260.10,243.38,277.91
1259,2026-05-26,259.65,242.04,279.73
1260,2026-05-27,259.59,241.95,276.81


In [51]:
future_forecast_summary.to_csv("apple_90_day_future_forecast.csv", index=False)

In [52]:
from google.colab import files

files.download("apple_90_day_future_forecast.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>